# Task note: label = participant ID (identification task)

In this notebook the **label is the participant ID**, parsed from the filename (`participantN`). This is therefore a participant **identification** task (who produced the recording), **not** gesture recognition.

Because the classes are the participants themselves, a participant-grouped split is impossible (it would hold whole classes out of training). Samples are split randomly per file, so every participant appears in every split by design. Results are not comparable to the `*gest*` notebooks.

Status: the code in this notebook has been changed (see the README "Known limitations") but has **not been re-run**; numbers from earlier runs are not comparable.

In [ ]:
# Reproducibility: one seed for every source of randomness used below.
import os
import random
import numpy as np

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

# torch is not used by the SVM pipeline; seed it only if it happens to be installed.
try:
    import torch
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
except ImportError:
    pass

# No DataLoader in this notebook. random_state=SEED is passed to every sklearn splitter / estimator below.

In [ ]:
import os
import scipy.io as sio
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score
import numpy as np
import glob

# Function to extract gesture label from the filename
def extract_participant_from_filename(filepath):
    filename = os.path.basename(filepath)
    parts = filename.split('_')
    for part in parts:
        if 'participant' in part:
            participant_id = part.replace('participant', '')
            return int(participant_id)  # Convert to integer
    return None  # If no participant ID found
# Function to load and process a single .mat file
def load_and_process_file(filepath, target_length):
    mat_data = sio.loadmat(filepath)
    data_emg = mat_data['data_emg']
    
    # Handle NaNs and Infinities
    data_emg = np.nan_to_num(data_emg, nan=0.0, posinf=0.0, neginf=0.0)
    
    # Pad or truncate the data to the target length
    if data_emg.shape[0] < target_length:
        padded_data = np.pad(data_emg, ((0, target_length - data_emg.shape[0]), (0, 0)), 'constant', constant_values=0)
    else:
        padded_data = data_emg[:target_length]
    
    # Flatten the data for SVM input
    return padded_data.flatten()

# Process a list of files and return processed data and labels
def process_files(file_list, target_length):
    X = []
    y = []
    
    for filepath in file_list:
        X.append(load_and_process_file(filepath, target_length))
        y.append(extract_participant_from_filename(filepath))
    
    return np.array(X), np.array(y)

# Path to the root folder where all .mat files are stored
root_folder = 's1_s2_s3_matfiles_5part'  # Update with your folder path

# Find all .mat files in the folder
all_mat_files = glob.glob(os.path.join(root_folder, '*.mat'))

# Collect all corresponding labels
all_labels = [extract_participant_from_filename(mat_file) for mat_file in all_mat_files]

# Target length for EMG signals (adjust based on your data)
target_length = 10240  # Modify this as needed

In [ ]:
# Split into train and test sets (nothing is fitted on the test set).
# Label = participant ID, so a participant-grouped split is not possible here (see the note at the top): random per-file split.
TEST_SIZE = 0.33

all_mat_files = np.array(all_mat_files)
all_labels = np.array(all_labels)
train_groups = None

train_idx, test_idx = train_test_split(np.arange(len(all_mat_files)), test_size=TEST_SIZE, random_state=SEED)
train_files = list(all_mat_files[train_idx])
test_files = list(all_mat_files[test_idx])
train_labels = list(all_labels[train_idx])
test_labels = list(all_labels[test_idx])

# Process training and test data
X_train, y_train = process_files(train_files, target_length)
X_test, y_test = process_files(test_files, target_length)
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin

class ChannelStandardizer(BaseEstimator, TransformerMixin):
    """Per-channel mean/std (over samples and time), fitted on the data passed to fit() only.
    X is (n_samples, target_length * n_channels), flattened from (time, channels).
    Zero-variance channels are guarded (std -> 1)."""
    def __init__(self, n_channels, eps=1e-8):
        self.n_channels = n_channels
        self.eps = eps

    def fit(self, X, y=None):
        X3 = np.asarray(X).reshape(X.shape[0], -1, self.n_channels)
        mean = np.zeros(self.n_channels)
        std = np.ones(self.n_channels)
        for c in range(self.n_channels):          # one channel at a time to limit memory
            col = X3[:, :, c]
            mean[c] = col.mean(dtype=np.float64)
            std[c] = col.std(dtype=np.float64)
        self.mean_ = mean
        self.std_ = np.where(std < self.eps, 1.0, std)
        return self

    def transform(self, X):
        X3 = np.asarray(X).reshape(X.shape[0], -1, self.n_channels)
        out = (X3 - self.mean_.astype(np.float32)) / self.std_.astype(np.float32)
        return out.reshape(X.shape[0], -1).astype(np.float32, copy=False)

n_channels = X_train.shape[1] // target_length
assert n_channels * target_length == X_train.shape[1], 'unexpected feature length'

pipe = Pipeline([
    ('channel_norm', ChannelStandardizer(n_channels=n_channels)),
    ('scaler', StandardScaler(copy=False)),
    ('pca', PCA(n_components=100, random_state=SEED)),   # adjust the number of components as needed
    ('svc', SVC(kernel='linear', C=1, gamma='scale')),
])

# Cross-validation on the training set only. The whole pipeline (channel normalisation, scaling, PCA, SVC)
# is re-fitted inside every fold, so no information from the validation fold reaches PCA / scaling.
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
cv_groups = None

cross_val_scores = cross_val_score(pipe, X_train, y_train, cv=cv, groups=cv_groups, scoring='accuracy')
print("Cross-Validation Accuracy Scores:", cross_val_scores)
print("Mean Cross-Validation Accuracy:", np.mean(cross_val_scores))

# Fit the full pipeline on the entire training set (test set untouched)
pipe.fit(X_train, y_train)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def report_eval(y_true, y_pred, labels, names=None, title='Test set'):
    """Accuracy, chance level, per-class accuracy and confusion matrix."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    labels = list(labels)
    names = [str(n) for n in (labels if names is None else names)]
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    support = cm.sum(axis=1)
    per_class = np.divide(cm.diagonal(), support, out=np.full(len(labels), np.nan), where=support > 0)
    acc = float((y_true == y_pred).mean())
    chance_uniform = 1.0 / len(labels)
    chance_majority = support.max() / support.sum()
    print(f'{title} accuracy: {acc:.4f}')
    print(f'Chance level: uniform 1/K = {chance_uniform:.4f} (K={len(labels)}), majority class = {chance_majority:.4f}')
    print('Per-class accuracy:')
    for n, a, s in zip(names, per_class, support):
        print(f'  class {n}: {a:.3f}  (n={s})')
    fig, axes = plt.subplots(1, 2, figsize=(max(10, 0.6 * len(labels) + 8), max(4, 0.35 * len(labels) + 3)))
    ConfusionMatrixDisplay(cm, display_labels=names).plot(ax=axes[0], colorbar=False, xticks_rotation=90)
    axes[0].set_title(f'{title}: confusion matrix')
    axes[1].bar(names, per_class)
    axes[1].axhline(chance_uniform, color='r', linestyle='--', label='uniform chance')
    axes[1].axhline(chance_majority, color='k', linestyle=':', label='majority-class chance')
    axes[1].set_ylim(0, 1)
    axes[1].set_xlabel('class')
    axes[1].set_ylabel('accuracy')
    axes[1].set_title(f'{title}: per-class accuracy')
    axes[1].legend()
    plt.setp(axes[1].get_xticklabels(), rotation=90)
    plt.tight_layout()
    plt.show()
    return acc, cm, per_class

# Final evaluation: the test set is used once, with the fitted pipeline.
y_pred = pipe.predict(X_test)
classes = np.unique(np.concatenate([y_train, y_test]))
test_acc, test_cm, test_per_class = report_eval(y_test, y_pred, labels=classes, title='Test set (SVM pipeline)')

In [ ]:
# # --- Hyperparameter Tuning with GridSearchCV (on the pipeline) ---

# from sklearn.model_selection import GridSearchCV

# param_grid = {
#     'svc__C': [0.01, 0.1, 1, 10, 100],
#     'svc__gamma': ['scale', 'auto', 0.01, 0.001, 0.0001],
#     'svc__kernel': ['rbf']
# }

# grid = GridSearchCV(pipe, param_grid, refit=True, verbose=2, cv=cv)
# grid.fit(X_train, y_train, groups=cv_groups)

# print("Best parameters:", grid.best_params_)
# y_pred_grid = grid.predict(X_test)
# print("Optimized Accuracy (test):", accuracy_score(y_test, y_pred_grid))